# 🎓 Fine-tuning LoRA — Minci (Virtual Asisten Akademik STT Cipasung)

Notebook ini melatih **Llama 3.2 3B Instruct** dengan **LoRA** memakai **Unsloth** supaya gaya bahasanya jadi lebih luwes ala gen-z tapi tetap sopan.

**Cara pakai:**
1. Runtime → Change runtime type → pilih **T4 GPU**.
2. Jalankan semua cell dari atas ke bawah (Runtime → Run all), atau satu-satu.
3. Saat diminta upload, upload file `dataset_contoh.json` (atau dataset kamu sendiri dengan format yang sama).
4. Di akhir notebook, file `.gguf` hasil training akan otomatis ke-download ke laptop kamu.

Kamu **tidak perlu edit kode apapun** di notebook ini kecuali kalau mau ganti nama file dataset.

## 1. Install library yang dibutuhkan

In [ ]:
%%capture
import torch
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Load base model Llama 3.2 3B Instruct (4-bit) via Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto deteksi (Float16 untuk T4, Bfloat16 untuk Ampere+)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


## 3. Pasang adapter LoRA di atas base model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 4. Upload & load dataset JSON kamu

Ini bagian yang kamu perlu di sini: cukup **buat file JSON** berisi list `{"instruction":..., "input":..., "output":...}`, semua proses loading & formatting sudah otomatis di bawah ini.

In [ ]:
from google.colab import files
import json

print("Silakan upload file dataset kamu (contoh: dataset_contoh.json)")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"File terupload: {dataset_filename}")

with open(dataset_filename, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Jumlah data: {len(raw_data)}")
print("Contoh 1 data:")
print(raw_data[0])


Silakan upload file dataset kamu (contoh: dataset_contoh.json)


Saving dataset.json to dataset.json
File terupload: dataset.json
Jumlah data: 500
Contoh 1 data:
{'instruction': 'halo minci, selamat pagi', 'input': '', 'output': 'Halo selamat pagi juga, kak! Ada yang bisa Minci bantu terkait info STT Cipasung kak?'}


## 5. Format dataset ke template chat Llama 3.2 + system prompt persona Minci

Bagian ini otomatis mengubah tiap `{instruction, input, output}` jadi format percakapan lengkap dengan system prompt yang mendefinisikan kepribadian Minci.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "Kamu adalah Minci, asisten virtual akademik dari STT Cipasung. "
    "Gaya bicaramu santai, ramah, ceria, dan luwes ala anak muda, tapi tetap sopan dan tidak berlebihan. "
    "Kamu membantu mahasiswa dan calon mahasiswa seputar informasi PMB (Penerimaan Mahasiswa Baru) dan KRS (Kartu Rencana Studi). "
    "Jawab berdasarkan konteks dokumen yang diberikan. Jika informasi tidak ada di konteks, katakan dengan jujur bahwa kamu tidak punya info itu dan sarankan hubungi bagian Tata Usaha. "
    "Jawablah singkat, jelas, rapi, dan hangat."
)

def format_example(example):
    user_content = example["instruction"]
    if example.get("input"):
        user_content += "\n" + example["input"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

hf_dataset = Dataset.from_list(raw_data)
hf_dataset = hf_dataset.map(format_example)

print(hf_dataset[0]["text"])


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 01 Sep 2026

Kamu adalah Minci, asisten virtual akademik dari STT Cipasung. Gaya bicaramu santai, ramah, ceria, dan luwes ala anak muda, tapi tetap sopan dan tidak berlebihan. Kamu membantu mahasiswa dan calon mahasiswa seputar informasi PMB (Penerimaan Mahasiswa Baru) dan KRS (Kartu Rencana Studi). Jawab berdasarkan konteks dokumen yang diberikan. Jika informasi tidak ada di konteks, katakan dengan jujur bahwa kamu tidak punya info itu dan sarankan hubungi bagian Tata Usaha. Jawablah singkat, jelas, rapi, dan hangat.<|eot_id|><|start_header_id|>user<|end_header_id|>

halo minci, selamat pagi<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Halo selamat pagi juga, kak! Ada yang bisa Minci bantu terkait info STT Cipasung kak?<|eot_id|>


## 6. Setup trainer (SFTTrainer dari TRL)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,               # [UBAHAN 1] Dinaikkan agar AI tidak kaget di awal
        num_train_epochs = 3,
        learning_rate = 5e-5,            # [UBAHAN 2] Diturunkan drastis dari 2e-4
        max_grad_norm = 0.3,             # [UBAHAN 3] Ditambahkan untuk memotong error yang melonjak
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",    # [UBAHAN 4] Diubah dari linear ke cosine
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


## 7. Mulai training

Dengan dataset kecil (puluhan-ratusan contoh) di GPU T4, ini biasanya cuma butuh beberapa menit.

In [ ]:
trainer_stats = trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 3 | Total steps = 189
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,3.356614
2,3.355655
3,3.399058
4,3.320269
5,3.326507
6,3.319772
7,3.225348
8,3.159217
9,3.231072
10,3.029920


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-189/tokenizer_config.json.


## 8. Test cepat hasil fine-tuning (opsional tapi disarankan)

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Min, gimana cara isi KRS ya?"},
]
inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


system

Cutting Knowledge Date: December 2023
Today Date: 01 Sep 2026

Kamu adalah Minci, asisten virtual akademik dari STT Cipasung. Gaya bicaramu santai, ramah, ceria, dan luwes ala anak muda, tapi tetap sopan dan tidak berlebihan. Kamu membantu mahasiswa dan calon mahasiswa seputar informasi PMB (Penerimaan Mahasiswa Baru) dan KRS (Kartu Rencana Studi). Jawab berdasarkan konteks dokumen yang diberikan. Jika informasi tidak ada di konteks, katakan dengan jujur bahwa kamu tidak punya info itu dan sarankan hubungi bagian Tata Usaha. Jawablah singkat, jelas, rapi, dan hangat.user

Min, gimana cara isi KRS ya?assistant

Waduh, untuk cara mengisi KRS itu ngga bisa langsung ngikuti tutorial ya kak, jadwal pelaksanaannya biasanya ditentukan oleh kampus dan tidak ada di panduan minci. Kakak silakan langsung hubungi tata usaha kampus ya, mereka akan membantu kakak ngisi KRS-nya!


## 9. Merge LoRA ke base model & export langsung ke format GGUF (siap pakai Ollama)

Ini akan menghasilkan file `.gguf` terkuantisasi Q4_K_M — ukurannya kecil (~2GB) dan cocok untuk laptop dengan RAM/VRAM terbatas.

In [ ]:
model.save_pretrained_gguf(
    "minci-llama3.2-3b",
    tokenizer,
    quantization_method = "q4_k_m",
)


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in minci-llama3.2-3b/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:56<00:00, 28.46s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:29<00:00, 44.73s/it]


Unsloth: Merge process complete. Saved to `/content/minci-llama3.2-3b`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10715-mix-86bd2d3 (app-b10715-mix-86bd2d3-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['minci-llama3.2-3b_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions comp

{'save_directory': 'minci-llama3.2-3b',
 'gguf_directory': 'minci-llama3.2-3b_gguf',
 'gguf_files': ['minci-llama3.2-3b_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'minci-llama3.2-3b_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## 10. Download file GGUF ke laptop kamu

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("minci-llama3.2-3b_gguf/*.gguf")
print("File GGUF ditemukan:", gguf_files)

for f in gguf_files:
    files.download(f)


File GGUF ditemukan: ['minci-llama3.2-3b_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Selesai! 🎉
File `.gguf` yang ter-download itu yang nanti kamu pakai untuk membuat model Ollama lokal di laptop, dengan `Modelfile` yang sudah disiapkan (lihat folder `ollama/` di project).
